# Image-Based Phenotyping & Machine Learning Pipeline
### Screening Sequential Drought and Salinity Tolerance in Rice (*Oryza sativa* L.)
**Authors:** Rifqi Alan Maulana et al.

---

## 📌 Overview
This notebook provides the complete, end-to-end reproducible workflow for:
1. **Multi-Stress Phenotypic Data Ingestion** (=148$ individual plants tracked across sequential drought and salinity stages).
2. **Feature Engineering**: Morphometric traits, vegetation indices (GLI, NGRDI), and dynamic inter-stage changes ($\Delta$ traits).
3. **Statistical Feature Selection**: Spearman rank correlation & Exploratory Factor Analysis with communality filtering (^2 \ge 0.50$).
4. **Leakage-Safe 5-Fold Stratified Cross-Validation**: Benchmarking 5 ML algorithms + Cumulative Odds Ordinal Logistic Regression.
5. **Granular Per-Class Breakdown** (Classes 1, 3, 5) & Publication-Ready Visualizations.


### 1. Environment Setup & Dependencies

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Add parent directory to path to load local package modules
sys.path.append(os.path.abspath('..'))

from src.data_loader import load_and_preprocess_data
from src.feature_selection import IBPFeatureSelector
from src.models import get_model_suite
from src.evaluation import (
    evaluate_models_cross_validation,
    generate_class_breakdown_table,
    plot_confusion_matrix_publication,
    plot_feature_importance_publication
)

print("✓ Libraries loaded successfully!")


### 2. Dataset Loading & Plant Identity Alignment
We load the image-based phenotyping (IBP) measurements recorded after progressive drought and subsequent salinity stress.
Row-for-row alignment ensures that each plant's sequential trajectory is tracked with zero identity mismatch.

In [ ]:
data = load_and_preprocess_data(data_dir="../data")
X = data['X']
X_all = data['X_all']
y_mts = data['y_mts']

print(f"Total Plants Evaluated: {len(X)}")
print(f"Static Features Extracted: {X.shape[1]}")
print(f"Total Features (including Dynamic Deltas): {X_all.shape[1]}")
print("
Target Class Distribution (MTS):
", pd.Series(y_mts).value_counts().sort_index())


### 3. Statistical Feature Selection
Features are screened through:
1. Spearman Rank Correlation with stress scores ( < 0.05$)
2. Factor Analysis with Kaiser criterion ($\text{eigenvalue} > 1$) and Varimax rotation
3. Communality thresholding (^2 \ge 0.50$), retaining traits accounting for $\ge 50\%$ of shared variance.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

selector = IBPFeatureSelector(communality_threshold=0.50)
X_selected = selector.fit_transform(X_scaled, y_mts)

print(f"✓ Retained {len(selector.selected_columns_)} high-communality traits:")
for feat in selector.selected_columns_:
    print(f"  - {feat} (Communality h² = {selector.communalities_[feat]:.3f})")


### 4. 5-Fold Stratified Cross-Validation Benchmarking (Table 5)
All models are evaluated via Stratified 5-Fold Cross-Validation, ensuring scaling and feature selection are fit strictly inside training folds.

In [ ]:
models = get_model_suite(random_state=42)
summary_df, oof_preds = evaluate_models_cross_validation(models, X, y_mts, n_splits=5, random_state=42)

print("
--- Table 5: 5-Model Multi-Stress Tolerance Benchmark ---")
display(summary_df)


### 5. Granular Per-Class Performance Breakdown (Table 6)
Evaluating model performance separately for:
- **Class 1**: Dual-Tolerant (Elite candidates)
- **Class 3**: Mono-Tolerant (Intermediate/moderate)
- **Class 5**: Dual-Susceptible (Eliminated)

In [ ]:
rf_oof = oof_preds['Random Forest']
table6_df = generate_class_breakdown_table(y_mts, rf_oof, model_name="Random Forest")

print("
--- Table 6: Per-Class Classification Report for Random Forest ---")
display(table6_df)


### 6. Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_mts, rf_oof, labels=[1, 3, 5])
classes = ['Class 1
(Dual-Tolerant)', 'Class 3
(Mono-Tolerant)', 'Class 5
(Dual-Susceptible)']

fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
cax = ax.matshow(cm, cmap='Blues', alpha=0.85)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        val = cm[i, j]
        color = "white" if val > cm.max() / 2 else "black"
        ax.text(j, i, str(val), ha="center", va="center", color=color, fontsize=12, fontweight="bold")
        
fig.colorbar(cax, shrink=0.8, pad=0.05, label="Number of Plants")
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(classes, fontsize=9, fontweight="bold")
ax.set_yticklabels(classes, fontsize=9, fontweight="bold")
ax.set_xlabel('Predicted MTS', fontsize=11, labelpad=10, fontweight="bold")
ax.set_ylabel('True MTS', fontsize=11, labelpad=10, fontweight="bold")
ax.set_title('Confusion Matrix: Random Forest (5-Fold CV)', fontsize=12, pad=15, fontweight="bold")
plt.tight_layout()
plt.show()


### 7. Feature Importance Ranking

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_full = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=42)
rf_full.fit(X_selected, y_mts)

df_imp = pd.DataFrame({'Feature': selector.selected_columns_, 'Importance': rf_full.feature_importances_})
df_imp = df_imp.sort_values(by='Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
y_pos = range(len(df_imp))
ax.barh(y_pos, df_imp['Importance'], color='#2b5c8f', edgecolor='black', height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_imp['Feature'], fontsize=9, fontweight="bold")
ax.set_xlabel('Gini Feature Importance', fontsize=11, fontweight="bold")
ax.set_title('Random Forest - Phenotypic Feature Importance Ranking', fontsize=12, pad=12, fontweight="bold")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
